In [71]:
import pandas as pd
import mysql.connector
from shapely.geometry import Polygon, shape
from shapely.wkt import loads as wkt_loads
from shapely import wkt
from shapely.geometry import Polygon

def fetch_blob_data(mysql_params):
    # Conectar a la base de datos
    conn = mysql.connector.connect(
        host=mysql_params["host"],
        port=mysql_params["port"],
        user=mysql_params["user"],
        password=mysql_params["password"],
        database=mysql_params["database"]
    )

    # Definir la consulta SQL
    query = """
    SELECT polygon
    FROM forest
    WHERE tipo_id = 21
    LIMIT 3
    """

    # Ejecutar la consulta y obtener los resultados
    cursor = conn.cursor()
    cursor.execute(query)
    
    # Convertir los resultados a un DataFrame
    df = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])

    # Cerrar la conexión a la base de datos
    cursor.close()
    conn.close()

    # Retornar el DataFrame con los resultados
    return df



def blob_to_df(shapely_objects):
    # Inicializar una lista para almacenar los arrays de coordenadas
    all_coords = []

    # Iterar sobre cada cadena de texto de un POLYGON
    for index, blob in enumerate(shapely_objects):
        print(f"\nProcesando blob {index + 1}/{len(shapely_objects)}:")
        print(f"Longitud de blob: {len(blob) if isinstance(blob, str) else 'No es una cadena'}")

        try:
            # Verificar si es un string con información WKT
            if isinstance(blob, str):
                print(f"Tipo de dato recibido: {type(blob)}")
                print(f"Contenido antes de limpiar: '{blob}'")

                # Limpiar el string recibido
                wkt_string = blob.strip().strip("b'").strip("'")  # Quitar b' y ' si es necesario
                print(f"Contenido después de limpiar: '{wkt_string}'")

                # Verificar si la cadena está vacía después de la limpieza
                if not wkt_string:
                    print(f"Cadena WKT vacía después de la limpieza: '{blob}'")
                    continue

                # Verificar si la cadena comienza con "POLYGON"
                if not wkt_string.lower().startswith("polygon"):
                    print(f"Cadena WKT no comienza con 'POLYGON': '{wkt_string}'")
                    continue

                # Convertir la cadena WKT a un objeto de Shapely
                shape_object = wkt.loads(wkt_string)
                print(f"Objeto Shapely creado: {shape_object}")

                # Verificar el tipo de objeto
                if isinstance(shape_object, Polygon):
                    # Obtener las coordenadas del POLYGON
                    coords = list(shape_object.exterior.coords)
                    print(f"Coordenadas extraídas: {coords}")

                    # Añadir coordenadas a la lista de todas las coordenadas
                    all_coords.append(coords)
                else:
                    print(f"El objeto no es un POLYGON: {shape_object}")
            else:
                print(f"Formato de entrada no reconocido: {blob}")

        except Exception as e:
            print(f"Error al procesar blob: {e}, entrada: {blob}")

    # Retornar la lista de arrays de coordenadas
    return all_coords



def detect_outliers(df):
    def find_outliers(series):
        Q1 = series.quantile(0.05)
        Q3 = series.quantile(0.95)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        print("lower bound is: ", lower_bound)
        print("upper bound is: ", upper_bound)
        return (series < lower_bound) | (series > upper_bound)
    
    # Detectar outliers en latitud y longitud
    lat_outliers_mask = find_outliers(df['Latitude'])
    lon_outliers_mask = find_outliers(df['Longitude'])
    
    # Combinar las máscaras de outliers para obtener las filas que son outliers en al menos una columna
    combined_outliers_mask = lat_outliers_mask | lon_outliers_mask
    
    # Filtrar el dataframe original para obtener solo las filas con outliers
    outliers = df[combined_outliers_mask]
    return outliers

In [72]:
from shapely.wkt import loads

# Parámetros de conexión a la base de datos
mysql_params = {
    "host": "database-forest.c1qoas008ih1.eu-north-1.rds.amazonaws.com",
    "port": 3306,
    "user": "admin",
    "password": "chetler2",
    "database": "forest-RDS"
}

df = fetch_blob_data(mysql_params)

df_2 = blob_to_df(df)




Procesando blob 1/3:
Longitud de blob: 7
Tipo de dato recibido: <class 'str'>
Contenido antes de limpiar: 'polygon'
Contenido después de limpiar: 'polygon'
Error al procesar blob: ParseException: Expected word but encountered end of stream, entrada: polygon


In [77]:
print(df.iloc[[0]])

                                             polygon
0  b'POLYGON ((-3.59881542274033 41.1106773391705...


In [73]:
print(f"Tipo de dato de la variable: {type(df)}")
print(f"Representación de la variable: {repr(df)}")

Tipo de dato de la variable: <class 'pandas.core.frame.DataFrame'>
Representación de la variable:                                              polygon
0  b'POLYGON ((-3.59881542274033 41.1106773391705...
1  b'POLYGON ((-3.907480449882828 40.845831593363...
2  b'POLYGON ((-3.5996148922555946 41.15501916573...


In [78]:
print(f"Tipo de dato de la variable: {type(df.iloc[[0]])}")
print(f"Representación de la variable: {repr(df.iloc[[0]])}")

Tipo de dato de la variable: <class 'pandas.core.frame.DataFrame'>
Representación de la variable:                                              polygon
0  b'POLYGON ((-3.59881542274033 41.1106773391705...


In [94]:
def clean_polygon_string(s):
    # Eliminar el prefijo "b'" y el sufijo "'"
    if isinstance(s, str):
        return s.strip("b'")  # También se puede usar strip() o replace() para remover estos caracteres
    return s

In [97]:
# Supongamos que tu DataFrame se llama df y la columna de interés es 'polygon'
df_2 = df['polygon'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

# Verificar el resultado
print(df_2[0])

POLYGON ((-3.59881542274033 41.11067733917059, -3.5989051052840915 41.110686205290676, -3.5989890503434485 41.11070355157339, -3.599074823535174 41.11069245094058, -3.5990946222367874 41.11063876571547, -3.5990968493757647 41.11057993010607, -3.5991130275582663 41.110583000579204, -3.5991501809571838 41.11062669523698, -3.5991969096580023 41.110660107898035, -3.5993014118985456 41.110665410538054, -3.5993243832300292 41.11063895976539, -3.5993471463869233 41.1105890964828, -3.5993507231446755 41.11055689229908, -3.5993246604750593 41.11052105894349, -3.599372073041053 41.11053550524002, -3.5994049759860642 41.110554467949804, -3.5994293713593644 41.110584941194944, -3.5994385243805467 41.110628808462906, -3.599451853067192 41.11065106183278, -3.5994738827034833 41.110668919064956, -3.599512692072118 41.11068277950773, -3.5998037251921255 41.11068215650685, -3.599898091541148 41.11067551362181, -3.599917907429324 41.11064511744406, -3.5999398804199263 41.11063305751989, -3.5999660925706

In [1]:
import matplotlib.pyplot as plt

coord = [[1,1], [2,1], [2,2], [1,2], [0.5,1.5]]
coord.append(coord[0]) #repeat the first point to create a 'closed loop'

xs, ys = zip(*df_2[0]) #create lists of x and y values

plt.figure()
plt.plot(xs,ys) 
plt.show() # if you need...

ModuleNotFoundError: No module named 'matplotlib'

In [101]:
pip install matplotlib.pyplot

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement matplotlib.pyplot (from versions: none)
ERROR: No matching distribution found for matplotlib.pyplot

[notice] A new release of pip is available: 23.2.1 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
coordinates = df.loc[0, 'coordinates']

In [46]:
# Crear el DataFrame
df = pd.DataFrame(coordinates, columns=["Longitude", "Latitude"])

# Mostrar el DataFrame
print(df)

     Longitude   Latitude
0    -3.598815  41.110677
1    -3.598905  41.110686
2    -3.598989  41.110704
3    -3.599075  41.110692
4    -3.599095  41.110639
..         ...        ...
234  -3.598765  41.110826
235  -3.598767  41.110768
236  -3.598804  41.110750
237  -3.598815  41.110719
238  -3.598815  41.110677

[239 rows x 2 columns]
